In [4]:
import pandas as pd
import os, dill
import numpy as np
import cvxpy as cp
from typing import Iterable, Optional, Mapping, cast
from plotly import graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from yaml import emit

os.chdir(os.path.expanduser('~/dev/vEcoli'))

%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
# import sim
time = '600'
date = '2026-04-06'
experiment = 'homeostatic_only'
condition = 'basal'
entry = f'{experiment}_{time}_{date}'
folder = f'out/objective_weight/{condition}/{entry}/'

output = np.load(folder + '0_output.npy',allow_pickle='TRUE').item()
output = output['agents']['0']
fba = output['listeners']['fba_results']
bulk = pd.DataFrame(output['bulk'])
f = open(folder + 'agent_steps.pkl', 'rb')
agent = dill.load(f)
f.close()

metabolism = agent['ecoli-metabolism-redux-classic']

In [6]:
# get objective terms across time
counts_to_molar = output['listeners']['enzyme_kinetics']['counts_to_molar']

homeostatic_term = np.asarray(fba['homeostatic_term'])/counts_to_molar #change to count
# secretion_term = np.asarray(fba['secretion_term'])/counts_to_molar
# efficiency_term = np.asarray(fba['efficiency_term'])/counts_to_molar
# kinetic_term = np.asarray(fba['kinetics_term'])/counts_to_molar
# diversity_term = np.asarray(fba['diversity_term'])/counts_to_molar

time = np.arange(len(homeostatic_term))

# objective_weights_sim = {
#             "secretion": 4.07e-10,
#             "efficiency": 8.56e-7,
#             "kinetics": 1.60e-3,
#             "diversity": 2.57e-7,
#             "homeostatic": 1.0,
# }

objective_weights_sim = {
            "secretion": 0,
            "efficiency": 0,  # decrease efficiency
            "kinetics": 0,  # 0.00001
            "diversity": 0,  # 0.001 Heena's addition to minimize number of reactions with no flow
            "homeostatic": 1,
        }

In [7]:
from plotly.colors import qualitative as q

spec_term = {
    "Homeostatic": (homeostatic_term, objective_weights_sim['homeostatic']),
    # "Secretion":   (secretion_term,   objective_weights_sim['secretion']),
    # "Efficiency":  (efficiency_term,  objective_weights_sim['efficiency']),
    # "Kinetic":     (kinetic_term,     objective_weights_sim['kinetics']),
    # "Diversity":     (diversity_term,     objective_weights_sim['diversity']),
}

spec_no_weights = {
    "Homeostatic": (homeostatic_term/objective_weights_sim['homeostatic'], objective_weights_sim['homeostatic']),
    # "Secretion":   (secretion_term/objective_weights_sim['secretion'],     objective_weights_sim['secretion']),
    # "Efficiency":  (efficiency_term/objective_weights_sim['efficiency'],   objective_weights_sim['efficiency']),
    # "Kinetic":     (kinetic_term/objective_weights_sim['kinetics'],        objective_weights_sim['kinetics']),
    # "Diversity":     (diversity_term/objective_weights_sim['diversity'],        objective_weights_sim['diversity']),
}


palette = q.Pastel  # consistent colors across both sets
colors = {name: palette[i % len(palette)] for i, name in enumerate(spec_term)}


fig = go.Figure()

# Add weighted (solid) + unweighted (dotted) per term
for i, (name, (y_w, w)) in enumerate(spec_term.items()):
    # solid, weighted
    fig.add_trace(go.Scatter(
        x=time, y=y_w, mode="lines",
        name=f"{name} (weighted)",
        line=dict(color=colors[name], dash="solid"),
        legendgroup=name,
        legendgrouptitle_text=name if i == 0 else None,
        hovertemplate=f"{name.lower()} weight={w}<br>{name.lower()} term value=%{{y:.4g}}<extra></extra>"
    ))
    # dotted, unweighted
    y_uw, w_uw = spec_no_weights[name]
    fig.add_trace(go.Scatter(
        x=time, y=y_uw, mode="lines",
        name=f"{name} (unweighted)",
        line=dict(color=colors[name], dash="dot"),
        legendgroup=name,
        hovertemplate=f"{name.lower()} (no weight)<br>{name.lower()} value=%{{y:.4g}}<extra></extra>"
    ))

fig.update_layout(
    title=f"Change of Objective Term Values (Counts) Over Sim. λkin = {objective_weights_sim['kinetics']}",
    xaxis_title="Time (s)",
    yaxis=dict(title="Objective Term Value", type="log", tickformat=".0e"),
    template="plotly_white",
    colorway=px.colors.qualitative.Pastel,
    legend=dict(title="Terms", groupclick="togglegroup"),  # click once to toggle both lines in a group
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)

fig.show(renderer="browser")
save_path = f'{folder}figures/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory '{save_path}' created")
fig.write_image(f'{save_path}WC_objective_terms.png', width=800, height=400, scale=3)
# fig.write_image(f'/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_secretion/'
#                 f'terms_kinetics_{objective_weights_sim['secretion']}.png', width=800, height=400, scale=3)

In [8]:
# find top 10 homeostatic loss
estimated_homeostatic_dmdt = pd.DataFrame(fba["estimated_homeostatic_dmdt"][1:], columns=metabolism.homeostatic_metabolites)
target_homeostatic_dmdt = pd.DataFrame(fba["target_homeostatic_dmdt"][1:], columns=metabolism.homeostatic_metabolites)
homeostatic_count = pd.DataFrame(np.asarray(fba["homeostatic_metabolite_counts"][1:]), columns = metabolism.homeostatic_metabolites)

# calculate individual metabolite objective loss
homeostatic_objective  = np.abs(target_homeostatic_dmdt - estimated_homeostatic_dmdt)/homeostatic_count

# find metabolites with most lost over time
met_loss_avg = homeostatic_objective.mean(axis=0)
time_loss_avg = homeostatic_objective.mean(axis=1)

temp = homeostatic_objective[time_loss_avg > 0]
temp = temp.loc[:, met_loss_avg > 0]
temp

,CPD-12261[p],glycogen-monomer[c]
0,0.000000,2.462472e-08
8,0.000224,0.000000e+00
9,0.000453,0.000000e+00
10,0.000689,0.000000e+00
11,0.000928,0.000000e+00
12,0.001171,0.000000e+00
13,0.001421,0.000000e+00
14,0.001675,0.000000e+00
15,0.001930,0.000000e+00
16,0.002187,0.000000e+00


In [9]:
def get_subset_S(S, met_of_interest):
    S_met = S.loc[met_of_interest, :]
    S_met = S_met.loc[:,~np.all(S_met == 0, axis=0)]
    return S_met, S_met.columns

In [10]:
stoichiometry = metabolism.stoichiometry.copy()
reaction_names = metabolism.reaction_names
metabolites = metabolism.metabolite_names.copy()
S = stoichiometry .copy()
S = pd.DataFrame(S, index=metabolites, columns=reaction_names)


S_met, rxns =  get_subset_S(S, temp.columns)
S_met

,RXN-11302,glycogen-monomer-extension
CPD-12261[p],1,0
glycogen-monomer[c],0,1


## These metabolites are both manually added for murein (CPD-12261) and glycogen. It's odd to see that flow can be sustained through these metabolites at some time steps but the need is n't met for some other timestep, even when homeosatic objective is the only objective present.

In [11]:
###----- 1. Visualize metabolite exchange -----###
met_of_interest = temp.columns

dmdt_diff = (target_homeostatic_dmdt - estimated_homeostatic_dmdt)/homeostatic_count
dmdt_diff_interest = dmdt_diff.loc[:,met_of_interest]
dmdt_diff_interest

,CPD-12261[p],glycogen-monomer[c]
0,0.0,-2.462472e-08
1,0.0,0.000000e+00
2,0.0,0.000000e+00
3,0.0,0.000000e+00
4,0.0,0.000000e+00
...,...,...
595,0.0,0.000000e+00
596,0.0,0.000000e+00
597,0.0,0.000000e+00
598,0.0,0.000000e+00


In [12]:
list(dmdt_diff_interest.columns)

['CPD-12261[p]', 'glycogen-monomer[c]']

In [13]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- 0) Compute unmet need for all homeostatic metabolites ---
dmdt_diff = (target_homeostatic_dmdt - estimated_homeostatic_dmdt) / homeostatic_count

# Optional: drop timepoints/columns with all-NaN or zeros if needed
dmdt_diff = dmdt_diff.replace([np.inf, -np.inf], np.nan).div(counts_to_molar[1:], axis='rows')

# --- 1) Pick top 5 metabolites by "biggest unmet need" ---
# Recommended metric: mean absolute unmet need over time
met_score = dmdt_diff.abs().mean(axis=0).sort_values(ascending=False)

top5_mets = met_score.head(5).index.tolist()

# If you instead want "most negative" (largest sustained deficit), use:
# met_score = dmdt_diff.mean(axis=0).sort_values()  # most negative first
# top5_mets = met_score.head(5).index.tolist()

top5_bar = met_score.loc[top5_mets]

# --- 2) Prepare bottom plot (your metabolites of interest) ---
met_of_interest = list(dmdt_diff_interest.columns)
df_line = dmdt_diff.loc[:, met_of_interest].copy()
df_line.index.name = "time_s"

long = (
    df_line.reset_index()
           .melt(id_vars="time_s", var_name="metabolite", value_name="unmet_need")
)

# --- 3) Consistent pastel colors (shared across both subplots) ---
pastel = px.colors.qualitative.Pastel
all_mets_for_colors = list(dict.fromkeys(top5_mets + met_of_interest))  # preserve order, unique
color_map = {m: pastel[i % len(pastel)] for i, m in enumerate(all_mets_for_colors)}

# --- 4) Make subplots: bar on top, lines on bottom ---
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.35, 0.65],
    vertical_spacing=0.12,
)

# Top: bar chart (histogram-like summary)
fig.add_trace(
    go.Bar(
        x=top5_bar.index,
        y=top5_bar.values,
        marker=dict(color=[color_map.get(m, pastel[0]) for m in top5_bar.index]),
        text = [f'{val:.1e}' for val in top5_bar.values],
        name="Unmet Homeostatic Need"
    ),
    row=1, col=1
)

# Bottom: time series for metabolites of interest
for met in met_of_interest:
    fig.add_trace(
        go.Scatter(
            x=long.loc[long["metabolite"] == met, "time_s"],
            y=long.loc[long["metabolite"] == met, "unmet_need"],
            mode="lines",
            name=met,
            line=dict(color=color_map.get(met, pastel[1]))
        ),
        row=2, col=1
    )


fig.update_yaxes(title_text="Unmet Need", row=1, col=1)

fig.update_xaxes(title_text="Time (s)", row=2, col=1)
fig.update_yaxes(title_text="L1_|Target - Estimate|", row=2, col=1)

fig.update_layout(
    title=f"Unmet homeostatic need summary + culprit metabolite need timeseries. λkin = {objective_weights_sim['kinetics']}",
    xaxis_title="Time (s)",
    yaxis=dict(title="Objective Term Value", type="log", tickformat=".0e"),
    template="plotly_white",
    colorway=px.colors.qualitative.Pastel,
    barcornerradius=15,
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)

fig.show(renderer='browser')

save_path = f'{folder}figures/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directoryy '{save_path}' created")

fig.write_image(f'{save_path}WC_metabolite_unmet_need.png', width=900, height=600, scale=3)


### glycogen-monomer[c] was made too much at timestep 1 (because diff is negative --> estimate > target). CPD-12261[p] is the main culprit for unmet need

In [14]:
###----- 2. Check if any of the reactions upstream of these metabolites have no enzyme, causing zero flow-----###
# 2.1 Run parsimonious FBA (pFBA) with to maximize CPD-12261 export to extract path of making CPD-12261
import cvxpy as cp
v = cp.Variable(len(reaction_names), nonneg=True)
e = cp.Variable(metabolism.network_flow_model.n_exch_rxns, nonneg=True)
dm = metabolism.network_flow_model.S_orig @ v + metabolism.network_flow_model.S_exch @ e
exch = metabolism.network_flow_model.S_exch @ e
upper_flux_bound = 100

constr = []
constr.append(dm[metabolism.network_flow_model.intermediates_idx] == 0)
constr.extend([v >= 0, v <= upper_flux_bound, e >= 0, e <= upper_flux_bound])
constr.extend([cp.sum(v) >= 10000])

# objective is to maximize dm[index_CPD-12261[p]] and minimize total flux
CPD_12261_index = metabolites.index('CPD-12261[p]')
objective = -dm[CPD_12261_index] + cp.sum(v)

problem = cp.Problem(cp.Minimize(objective), constr)
problem.solve(solver=cp.GLOP)


np.float64(9975.0)

In [15]:
solution = pd.DataFrame(v.value, index=reaction_names)
homeostatic_metabolite_idx = [metabolites.index(met) for met in metabolism.homeostatic_metabolites]
dm_solution = pd.DataFrame(dm[homeostatic_metabolite_idx].value, index=metabolism.homeostatic_metabolites)
reactions_making_CPD12261 = solution.loc[solution[0] != 0].index

reaction_catalyst_counts_all_time = pd.DataFrame(fba['reaction_catalyst_counts'][1:], columns=reaction_names)
reaction_catalyst_counts_CPD12261 = reaction_catalyst_counts_all_time.loc[:, reactions_making_CPD12261]

In [16]:
# UDP-NACMURALA-GLU-LIG-RXN is the only reaction with occasional zero enzyme counts
# 2.2 Check to see if zero enzyme counts of UDP-NACMURALA-GLU-LIG-RXN coincides with that of unmet CPD-12261 need
# enzyme is UDP-NACMURALA-GLU-LIG-MONOMER
reaction = "UDP-NACMURALA-GLU-LIG-RXN"

enzyme_timeseries = reaction_catalyst_counts_CPD12261.loc[:, reaction]
enzyme_name = metabolism.parameters['reaction_catalysts'][reaction]

In [17]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

met = "CPD-12261[p]"  # (you wrote CPD-12251[p] once—use the exact column name you have)
enzyme_label = f"{enzyme_name} (counts)"
need_label = f"{met} unmet need (normalized dmdt diff)"

# --- align indices (time) robustly ---
enz = enzyme_timeseries.copy()
need = dmdt_diff_interest[met].copy() if isinstance(dmdt_diff_interest, pd.DataFrame) else dmdt_diff_interest
pastel = px.colors.qualitative.Pastel
# Ensure both are Series with numeric index
enz = pd.Series(enz.values, index=enz.index, name="enzyme_counts")
need = pd.Series(need.values, index=need.index, name="unmet_need")

# Inner-join on shared timepoints
df_plot = pd.concat([enz, need], axis=1, join="inner").dropna()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["enzyme_counts"],
        mode="lines",
        name=enzyme_label,
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=df_plot.index,
        y=df_plot["unmet_need"],
        mode="lines",
        name=need_label,
    ),
    secondary_y=True
)

fig.update_layout(
    title=f"Enzyme availability of UDP-NACMURALA-GLU-LIG-RXN vs unmet need for {met}. λdiv = {objective_weights_sim['diversity']}",
    template="plotly_white",
    legend_title_text="Trace",
    colorway=px.colors.qualitative.Pastel,
    paper_bgcolor='rgba(255, 0, 0, 0)',
    plot_bgcolor='rgba(255, 0, 0, 0)',
)

fig.update_xaxes(title_text="Time (s)")
fig.update_yaxes(title_text="Enzyme counts", secondary_y=False)
fig.update_yaxes(title_text="L1_|Target - Estimate|", secondary_y=True)

fig.show(renderer="browser")
save_path = f'{folder}figures/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Directory '{save_path}' created")

fig.write_image(f'{save_path}WC_{met}_enzyme_availiability.png', width=1200, height=500, scale=3)

# Side 1: See if any reactions leading to CPD-12261 is kinetically constrained

In [20]:
kinetic_reactions = np.array(metabolism.kinetic_constraint_reactions)
isin = np.isin(reactions_making_CPD12261, kinetic_reactions)

kinetically_constrained_CPD12261_reactions = reactions_making_CPD12261[isin]

In [21]:
[reaction_names.index(reaction) for reaction in kinetically_constrained_CPD12261_reactions]

[914, 1164, 1406, 1411, 9344, 9349, 9358, 9365]

In [22]:
reaction

'UDP-NACMURALA-GLU-LIG-RXN'

In [23]:
flux_zero_once

NameError: name 'flux_zero_once' is not defined

In [24]:
# check flux of these reactions to see if that's why CPD-12261 isn't made
sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names)
flux_of_interest = sim_flux.loc[:, reactions_making_CPD12261]

flux_zero_once = reactions_making_CPD12261[np.any(flux_of_interest == 0, axis=0)]

flux_of_interest = sim_flux.loc[:, flux_zero_once]

In [25]:
# plot flux of these 8 reactions

# 1) Make an explicit time column (use the index if it already represents time)
plot_df = flux_of_interest.copy()

# If your index is just 0..T-1 and that’s fine as “timestep”, keep it:
plot_df["timestep"] = plot_df.index

# 2) Long format for Plotly Express
long_df = plot_df.melt(
    id_vars="timestep",
    var_name="reaction",
    value_name="flux",
)

# 3) Plot
fig = px.line(
    long_df,
    x="timestep",
    y="flux",
    color="reaction",
    title="Flux over time (8 reactions)",

)

fig.update_layout(
    template="plotly_white",
    legend_title_text="Reaction",
)

fig.show(renderer="browser")

# Side 2: plot unmet need for CPD16621

In [44]:
# find top 10 homeostatic loss
estimated_homeostatic_dmdt = pd.DataFrame(fba["estimated_homeostatic_dmdt"][1:], columns=metabolism.homeostatic_metabolites)
target_homeostatic_dmdt = pd.DataFrame(fba["target_homeostatic_dmdt"][1:], columns=metabolism.homeostatic_metabolites)

estimated_CPD12261 = estimated_homeostatic_dmdt.loc[:, "CYTIDINE[c]"]
target_CPD12261 = target_homeostatic_dmdt.loc[:,"CYTIDINE[c]"]

In [45]:
df = pd.DataFrame({
    "timestep": estimated_CPD12261.index,
    "estimated": estimated_CPD12261.values,
    "target": target_CPD12261.values,
})

long = df.melt(
    id_vars="timestep",
    value_vars=["estimated", "target"],
    var_name="series",
    value_name="value",
)

fig = px.line(
    long,
    x="timestep",
    y="value",
    color="series",
    title="CYTIDINE[c]: estimated vs target homeostatic term",
    labels={"value": "dM/dt (or homeostatic term)", "series": ""},
)

fig.update_layout(
    template="plotly_white",
    hovermode="x unified",
)

fig.show()



# Side 3: Find out bottleneck reaction. One that caps RXN-11302 at it's max flux

In [28]:
# 1) Extract
flux_of_interest = sim_flux.loc[:, kinetically_constrained_CPD12261_reactions]

target_reaction_flux = pd.DataFrame(fba["target_kinetic_fluxes"], columns=kinetic_reactions)
target_of_interest = target_reaction_flux.loc[:, kinetically_constrained_CPD12261_reactions]

# 2) Ensure the two have the same index (timestep alignment)
target_of_interest = target_of_interest.set_index(flux_of_interest.index) \
    if not target_of_interest.index.equals(flux_of_interest.index) else target_of_interest

t = flux_of_interest.index

# 3) Long format
est_long = (
    flux_of_interest
    .assign(timestep=t)
    .melt(id_vars="timestep", var_name="reaction", value_name="flux")
)
est_long["kind"] = "estimated"

tar_long = (
    target_of_interest
    .assign(timestep=t)
    .melt(id_vars="timestep", var_name="reaction", value_name="flux")
)
tar_long["kind"] = "target"

plot_df = pd.concat([est_long, tar_long], ignore_index=True)

# 4) Plot: same color per reaction, dashed by kind
fig = px.line(
    plot_df,
    x="timestep",
    y="flux",
    color="reaction",
    line_dash="kind",
    title="Estimated vs Target flux for CPD-12261-related kinetic reactions",
    labels={"flux": "Flux", "reaction": "Reaction", "kind": ""},
)

# Make styles exactly solid vs dotted (optional but nice)
fig.update_traces()  # no-op, just here if you want t_

fig.show(renderer="browser")

In [29]:
max_CPD12261_reaction = np.max(sim_flux.loc[:,"RXN-11302"])
max_CPD12261_reaction

np.float64(2931.0)

In [30]:
np.max(sim_flux.loc[:,"TRANS-RXN0-286"])

np.float64(11726.0)

In [31]:
met_of_interest = ["C6[c]"]
S_met, rxn = get_subset_S(S, met_of_interest)
rxn = ['UDPGLCNACEPIM-RXN', 'UDPGLCNACEPIM-RXN (reverse)', 'TRANS-RXN0-286', 'RXN0-5405']

flux_of_interest = sim_flux.loc[:, rxn]
flux_of_interest

,UDPGLCNACEPIM-RXN,UDPGLCNACEPIM-RXN (reverse),TRANS-RXN0-286,RXN0-5405
0,NaN,NaN,NaN,NaN
1,298.0,298.0,2.0,1.0
2,298.0,298.0,263.0,132.0
3,298.0,298.0,460.0,230.0
4,298.0,298.0,607.0,304.0
...,...,...,...,...
596,298.0,298.0,0.0,0.0
597,298.0,298.0,0.0,0.0
598,298.0,298.0,0.0,0.0
599,298.0,298.0,0.0,0.0


In [43]:
sim_flux_long = sim_flux.melt(var_name="reaction", value_name="flux")
fig = px.histogram(
    sim_flux_long,
    x="flux",
    color="reaction",
    title="Flux Distribution (all reactions)"
)
fig.show(renderer="browser")

In [33]:
kinetically_constrained_CPD12261_reactions

Index(['DIAMINOPIMEPIM-RXN', 'GLUTRACE-RXN', 'NACGLCTRANS-RXN',
       'NADH-DEHYDROG-A-RXN-NADH/UBIQUINONE-8/PROTON//NAD/CPD-9956/PROTON.46.',
       'UDP-NACMUR-ALA-LIG-RXN', 'UDPGLCNACEPIM-RXN',
       'UDPNACETYLGLUCOSAMENOLPYRTRANS-RXN', 'UGD-RXN'],
      dtype='object')

In [34]:
["NACGLCTRANS-RXN", 'UDP-NACMUR-ALA-LIG-RXN','UDPNACETYLGLUCOSAMENOLPYRTRANS-RXN']

['NACGLCTRANS-RXN',
 'UDP-NACMUR-ALA-LIG-RXN',
 'UDPNACETYLGLUCOSAMENOLPYRTRANS-RXN']

In [35]:
np.max(sim_flux.loc[:,"NAG1P-URIDYLTRANS-RXN"])

np.float64(26161.0)

In [36]:
met = ["UDP-N-ACETYL-D-GLUCOSAMINE[c]"]

S_met, rxn = get_subset_S(S, met)
temp = sim_flux.loc[169,rxn]

temp['kinetic'] = [target_reaction_flux[r] for r in temp.index if r in kinetic_reactions else False]
temp

SyntaxError: invalid syntax (4076915669.py, line 6)

In [37]:
flux_of_interest = sim_flux.loc[:, kinetically_constrained_CPD12261_reactions]
boo = flux_of_interest.max(axis=0) <= max_CPD12261_reaction
kinetically_constrained_CPD12261_reactions[boo]

Index(['UDPGLCNACEPIM-RXN', 'UGD-RXN'], dtype='object')

In [38]:
"UNDECAPRENYL-DIPHOSPHATE[c]" in metabolism.homeostatic_metabolites

True

In [39]:
# find top 10 homeostatic loss
estimated_homeostatic_dmdt = pd.DataFrame(fba["estimated_homeostatic_dmdt"][1:], columns=metabolism.homeostatic_metabolites)
target_homeostatic_dmdt = pd.DataFrame(fba["target_homeostatic_dmdt"][1:], columns=metabolism.homeostatic_metabolites)

estimated_other = estimated_homeostatic_dmdt.loc[:,"UNDECAPRENYL-DIPHOSPHATE[c]"]
target_other = target_homeostatic_dmdt.loc[:, "UNDECAPRENYL-DIPHOSPHATE[c]"]

In [40]:
import pandas as pd
import plotly.express as px

def plot_reaction_fluxes(
    sim_flux: pd.DataFrame,
    reaction_names,
    kinetic_reactions,
    target_reaction_flux: pd.DataFrame | None = None,
    title: str | None = None,
):
    """
    Plot estimated flux for each reaction in reaction_names over time.
    If a reaction is in kinetic_reactions AND target_reaction_flux is provided,
    also plot its target flux in the same color with a dotted line.

    Parameters
    ----------
    sim_flux : pd.DataFrame
        Rows = timesteps, columns = reactions, values = estimated flux.
    reaction_names : list[str] | np.ndarray
        Reactions to plot.
    kinetic_reactions : set/list[str]
        Reactions that have kinetic targets.
    target_reaction_flux : pd.DataFrame, optional
        Rows = timesteps, columns = reactions, values = target flux.
        Must share the same index as sim_flux (or be alignable).
    title : str, optional
        Figure title.
    """
    rxns = [r for r in reaction_names if r in sim_flux.columns]
    missing = [r for r in reaction_names if r not in sim_flux.columns]
    if not rxns:
        raise ValueError("None of the requested reactions are present in sim_flux.columns.")
    if missing:
        print(f"Warning: {len(missing)} reactions not found in sim_flux (ignored): {missing[:10]}")

    kinetic_set = set(kinetic_reactions)
    plot_rxns_kin = [r for r in rxns if r in kinetic_set]

    # --- estimated long ---
    est_long = (
        sim_flux.loc[:, rxns]
        .reset_index(names="timestep")
        .melt(id_vars="timestep", var_name="reaction", value_name="flux")
    )
    est_long["kind"] = "estimated"

    frames = [est_long]

    # --- target long (only for kinetic reactions, if provided) ---
    if target_reaction_flux is not None and len(plot_rxns_kin) > 0:
        # align index to sim_flux
        target_aligned = target_reaction_flux.copy()
        if not target_aligned.index.equals(sim_flux.index):
            target_aligned = target_aligned.reindex(sim_flux.index)

        tar_long = (
            target_aligned.loc[:, [r for r in plot_rxns_kin if r in target_aligned.columns]]
            .reset_index(names="timestep")
            .melt(id_vars="timestep", var_name="reaction", value_name="flux")
        )
        tar_long["kind"] = "target"
        frames.append(tar_long)

    plot_df = pd.concat(frames, ignore_index=True)

    pastel = px.colors.qualitative.Pastel

    fig = px.line(
        plot_df,
        x="timestep",
        y="flux",
        color="reaction",       # same color per reaction
        color_discrete_sequence=pastel,
        line_dash="kind",       # estimated vs target style
        title=title or "Reaction flux over time (estimated vs target where available)",
        labels={"flux": "Flux", "reaction": "Reaction", "kind": ""},
    )

    # Force dash styles exactly
    fig.update_traces()  # no-op placeholder
    fig.update_layout(template="plotly_white", hovermode="x unified", legend_title_text="Reaction")

    # Plotly will create legend entries like "RXN, estimated" and "RXN, target"
    # We'll standardize dash mapping by trace name suffix:
    fig.for_each_trace(lambda tr: tr.update(line=dict(dash="solid")) if tr.name.endswith(", estimated") else None)
    fig.for_each_trace(lambda tr: tr.update(line=dict(dash="dot")) if tr.name.endswith(", target") else None)

    return fig


In [41]:
reactions = ["NACGLCTRANS-RXN", "NACGLCTRANS-RXN (reverse)",
             "UDPNACETYLGLUCOSAMENOLPYRTRANS-RXN", "UDPNACETYLGLUCOSAMENOLPYRTRANS-RXN (reverse)",
             "NAG1P-URIDYLTRANS-RXN"]
fig = plot_reaction_fluxes(
    sim_flux=sim_flux,
    reaction_names=reactions,
    kinetic_reactions=kinetic_reactions,
    target_reaction_flux=target_reaction_flux,
    title="CPD-12261-associated reactions: estimated vs target flux",
)
fig.show(renderer="browser")

In [31]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

def sankey_metabolite_flux(
    met_of_interest,
    S: pd.DataFrame,
    flux,
    *,
    timestep=None,
    agg: str = "mean",
    top_n: int = 12,
    min_abs_contrib: float | None = None,
    title: str | None = None,
):
    """
    Sankey for ONE metabolite using user's get_subset_S().

    Nodes:
      producer reactions -> metabolite -> consumer reactions

    Link weights:
      |S_met,rxn * v_rxn|  (contribution magnitude)

    met_of_interest : str or list[str]
        Metabolite name (or a single-item list) matching S.index.
    S : pd.DataFrame
        Stoichiometry matrix (mets x rxns).
    flux : pd.Series or pd.DataFrame
        Estimated fluxes. Series: rxn->v. DataFrame: time x rxn.
    timestep : optional
        If flux is DataFrame, pick a single timestep row. If None, aggregate by `agg`.
    agg : {"mean","sum","median","max","min"}
        Aggregation if timestep is None.
    top_n : int
        Keep top_n producers and top_n consumers by |S*v|.
    min_abs_contrib : float, optional
        Filter out links with |S*v| below this.
    title : str, optional
        Figure title.

    Returns
    -------
    fig : plotly.graph_objects.Figure
    contrib_table : pd.DataFrame
    """

    # --- user's helper ---
    def get_subset_S(S, met_of_interest):
        S_met = S.loc[met_of_interest, :]
        S_met = S_met.loc[:, ~np.all(S_met == 0, axis=0)]
        return S_met, S_met.columns

    # Normalize met input
    if isinstance(met_of_interest, str):
        met_list = [met_of_interest]
    else:
        met_list = list(met_of_interest)

    if len(met_list) != 1:
        raise ValueError("Please pass exactly one metabolite name (string or single-item list).")
    met = met_list[0]

    # Get stoich row subset (1 x rxns) and participating rxns
    S_met, rxns = get_subset_S(S, [met])  # returns DataFrame with 1 row
    s = S_met.loc[met]                    # Series indexed by rxns

    # --- pick a single flux vector v (Series indexed by rxns) ---
    if isinstance(flux, pd.DataFrame):
        # restrict to participating rxns first to avoid reindex noise
        flux_sub = flux.loc[:, rxns]

        if timestep is not None:
            v = flux_sub.loc[timestep]
        else:
            if agg == "mean":
                v = flux_sub.mean(axis=0)
            elif agg == "sum":
                v = flux_sub.sum(axis=0)
            elif agg == "median":
                v = flux_sub.median(axis=0)
            elif agg == "max":
                v = flux_sub.max(axis=0)
            elif agg == "min":
                v = flux_sub.min(axis=0)
            else:
                raise ValueError(f"Unknown agg='{agg}'")
    elif isinstance(flux, pd.Series):
        v = flux.reindex(rxns).fillna(0.0)
    else:
        raise TypeError("flux must be a pandas Series or DataFrame")

    # Signed contribution to d(met)/dt
    contrib = s * v
    abs_contrib = contrib.abs()

    contrib_table = pd.DataFrame(
        {
            "stoich": s,
            "flux": v,
            "contrib": contrib,
            "abs_contrib": abs_contrib,
        }
    ).sort_values("abs_contrib", ascending=False)

    if min_abs_contrib is not None:
        contrib_table = contrib_table[contrib_table["abs_contrib"] >= float(min_abs_contrib)]

    # Producers/consumers by SIGN of contribution
    producers = contrib_table[contrib_table["contrib"] > 0].head(top_n)
    consumers = contrib_table[contrib_table["contrib"] < 0].head(top_n)

    if producers.empty and consumers.empty:
        raise ValueError(
            f"No nonzero contributions found for '{met}' "
            f"(after filtering). Try a different timestep/agg or lower min_abs_contrib."
        )

    # --- Sankey nodes ---
    prod_names = producers.index.tolist()
    cons_names = consumers.index.tolist()
    nodes = prod_names + [met] + cons_names
    idx = {name: i for i, name in enumerate(nodes)}
    met_idx = idx[met]

    # --- Sankey links ---
    sources, targets, values, link_labels = [], [], [], []

    for r, row in producers.iterrows():
        sources.append(idx[r])
        targets.append(met_idx)
        values.append(float(row["abs_contrib"]))
        link_labels.append(f"{r} produces {met}: |S*v|={row['abs_contrib']:.3g}")

    for r, row in consumers.iterrows():
        sources.append(met_idx)
        targets.append(idx[r])
        values.append(float(row["abs_contrib"]))
        link_labels.append(f"{r} consumes {met}: |S*v|={row['abs_contrib']:.3g}")

    # --- Plot ---
    fig = go.Figure(
        data=[
            go.Sankey(
                arrangement="snap",
                node=dict(
                    pad=12,
                    thickness=16,
                    line=dict(width=0.5),
                    label=nodes,
                ),
                link=dict(
                    source=sources,
                    target=targets,
                    value=values,
                    label=link_labels,
                ),
            )
        ]
    )

    if title is None:
        if isinstance(flux, pd.DataFrame) and timestep is not None:
            title = f"{met}: production/consumption contributions (t={timestep})"
        elif isinstance(flux, pd.DataFrame):
            title = f"{met}: production/consumption contributions ({agg} over time)"
        else:
            title = f"{met}: production/consumption contributions"

    fig.update_layout(title=title, template="plotly_white", height=650)
    return fig, contrib_table


In [46]:
met_of_interest = ['CYTIDINE[c]']
S_met, rxn = get_subset_S(S, met_of_interest)
rxn

Index(['CYTIDEAM2-RXN', 'CYTIDINEKIN-RXN', 'CYTIKIN-RXN',
       'RXN-14090[CCO-CYTOSOL]-CPD-3711/WATER//CYTIDINE/Pi.41.', 'RXN-21270',
       'RXN0-361', 'TRANS-RXN-108-CYTIDINE/PROTON//CYTIDINE/PROTON.33.',
       'TRANS-RXN-108B'],
      dtype='object')

In [52]:
sim_flux.loc[500, rxn][0]

/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_62109/3040323529.py:1: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`



np.float64(0.0)

In [51]:
sim_flux = pd.DataFrame(fba['estimated_fluxes'], columns=reaction_names)
fig, tab = sankey_metabolite_flux(
    met_of_interest="CYTIDINE[c]",
    S=S,
    flux=sim_flux,
    agg="max",
    top_n=12,
    min_abs_contrib=-1,
)
fig.show()


ValueError: No nonzero contributions found for 'CYTIDINE[c]' (after filtering). Try a different timestep/agg or lower min_abs_contrib.

In [44]:
fig, tab = sankey_metabolite_flux(
    met_of_interest="UDP-N-ACETYL-D-GLUCOSAMINE[c]",
    S=S,
    flux=sim_flux,
    agg="mean",
    top_n=12,
    min_abs_contrib=1e-3,
)

fig.update_layout(
    colorway=px.colors.qualitative.Pastel,
    # paper_bgcolor='rgba(255, 0, 0, 0)',
    # plot_bgcolor='rgba(255, 0, 0, 0)',
)
fig.show()
# fig.write_image(f"/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/CPD12661_kinetics_analysis/"
#                 f"UDP-N-ACETYL-D-GLUCOSAMINE_production_consumption.png",
#                 width=1000, height=500, scale=3)


In [45]:
reactions = ["UDPNACETYLGLUCOSAMACYLTRANS-RXN",
             "NACGLCTRANS-RXN",
             "UDPNACETYLGLUCOSAMENOLPYRTRANS-RXN",
             "RXN-19737",
             "UDPGLCNACEPIM-RXN"]

fig = plot_reaction_fluxes(
    sim_flux=sim_flux,
    reaction_names=reactions,
    kinetic_reactions=kinetic_reactions,
    target_reaction_flux=target_reaction_flux,
    title="CPD-12261-competing reactions: estimated vs target flux",
)

fig.update_layout(
    colorway=px.colors.qualitative.Pastel,
    # paper_bgcolor='rgba(255, 0, 0, 0)',
    # plot_bgcolor='rgba(255, 0, 0, 0)',
)
fig.show(renderer='browser')
# fig.write_image(f"/Users/HeenaSaqib/dev/vEcoli/notebooks/Heena notebooks/Metabolism_New Genes/out/objective_weights/homeostatic_kinetics/CPD12661_kinetics_analysis/"
#                 f"UDP-N-ACETYL-D-GLUCOSAMINE_competing_reaction_fluxes.png",
#                 width=1200, height=500, scale=3)


In [46]:
fig, tab = sankey_metabolite_flux(
    met_of_interest="GLUCOSAMINE-1P[c]",
    S=S,
    flux=sim_flux,
    agg="mean",
    top_n=12,
    min_abs_contrib=1e-3,
)
fig.show()

# Side 3: Check secretion metabolites

In [15]:
metabolism.homeostatic_metabolites

array(['2-3-DIHYDROXYBENZOATE[c]', '2-KETOGLUTARATE[c]', '2-PG[c]',
       '2K-4CH3-PENTANOATE[c]', '4-AMINO-BUTYRATE[c]',
       '4-hydroxybenzoate[c]', 'ACETOACETYL-COA[c]', 'ACETYL-COA[c]',
       'ACETYL-P[c]', 'ADENINE[c]', 'ADENOSINE[c]', 'ADP-D-GLUCOSE[c]',
       'ADP[c]', 'AMP[c]', 'ANTHRANILATE[c]', 'APS[c]', 'ARG[c]',
       'ASN[c]', 'ATP[c]', 'BIOTIN[c]', 'CA+2[c]', 'CAMP[c]',
       'CARBAMYUL-L-ASPARTATE[c]', 'CARBON-DIOXIDE[c]', 'CDP[c]',
       'CHORISMATE[c]', 'CIS-ACONITATE[c]', 'CIT[c]', 'CL-[c]', 'CMP[c]',
       'CO+2[c]', 'CO-A[c]', 'CPD-12115[c]', 'CPD-12261[p]',
       'CPD-12575[c]', 'CPD-12819[c]', 'CPD-12824[c]', 'CPD-13469[c]',
       'CPD-2961[c]', 'CPD-8260[c]', 'CPD-9956[c]', 'CPD0-939[c]',
       'CTP[c]', 'CYS[c]', 'CYTIDINE[c]', 'CYTOSINE[c]', 'D-ALA-D-ALA[c]',
       'D-SEDOHEPTULOSE-7-P[c]', 'DAMP[c]', 'DATP[c]', 'DCTP[c]',
       'DEOXY-RIBOSE-5P[c]', 'DEOXYADENOSINE[c]', 'DEOXYGUANOSINE[c]',
       'DGMP[c]', 'DGTP[c]', 'DI-H-OROTATE[c]',
       '